In [1]:
import sys
sys.path.append('/Users/rifatordulu/Developer/lstm-stock-price-prediction')
# sys.path.append('/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers')

%run "../helpers/data_manipulator_v2.py"
%run "../helpers/yfinance_data_fetcher.py"

from custom_objects import register_custom_objects
# Register globally
register_custom_objects()

### LOAD VECTORS FOR TODAY

In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler
from keras.models import load_model

combos = [
    ["ABNB", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["ABNB", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["ABNB", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["ABNB", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["AMD", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["AMD", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["AMZN", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["AMZN", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["AVGO", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["BA", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["BA", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["BA", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["BA", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 4],
    ["BA", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 5],
    ["BA", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 4],
    ["BA", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["BABA", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["BABA", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["BABA", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 3],
    ["BAC", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 4],
    ["BAC", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["CRM", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["CRM", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["CSCO", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 4],
    ["CSCO", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["CSCO", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["DIS", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["DIS", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["DIS", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["DIS", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["DIS", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["GOOGL", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["GOOGL", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["GOOGL", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["GOOGL", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["GOOGL", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["INTC", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["INTC", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["JNJ", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["JNJ", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["JNJ", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["JPM", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["JPM", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["JPM", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["MSFT", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["MSFT", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 3],
    ["MU", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["MU", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["MU", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["MU", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 3],
    ["MU", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 5],
    ["NFLX", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["NVDA", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["NVDA", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["NVDA", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["NVDA", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 4],
    ["NVDA", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["PFE", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["PFE", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["PLTR", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 4],
    ["PLTR", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 4],
    ["QQQ", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["QQQ", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["RGTI", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 4],
    ["RGTI", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["SHOP", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["T", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["T", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["TSLA", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["TSLA", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["TSLA", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["UBER", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["UBER", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["UBER", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 3],
    ["UBER", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["UNH", "experimental", "Next-Day-Close-To-Next-Day-Open-Ratio", 4],
    ["UNH", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["UNH", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["UNH", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["UNH", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["V", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["WFC", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["WFC", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["WFC", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["WMT", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["WMT", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 3],
    ["WMT", "complex", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["WMT", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["XOM", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 4],
    ["MSTR", "complex", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    ["MSTR", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["MSTR", "simple", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["MSTR", "experimental", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["MSTR", "medium", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["AAPL", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 1],
    ["AAPL", "simple", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["DASH", "medium", "Next-Day-Close-To-Next-Day-Open-Ratio", 2],
    
                   ]


write_header = True
for combo in combos:
    ss = combo[0]
    mm = combo[1]
    tt = combo[2]
    print(f"SS: {ss}, MM: {mm} TT: {tt}")
    score = combo[3]
    # Prepare model location and name
    model_location = "../models-feb-update/" + ss + "/cnn/" + mm + "/"
    model_name = tt + ".keras"
    threshold = pd.read_csv(model_location + tt + "no-quantile_threshold.csv").iloc[0]["Threshold"]
    lvls = pd.read_csv(model_location + tt + "no-quantile_all_thresholds.csv")
    threshold_levels = lvls.iloc[(lvls['threshold'] - threshold).abs().idxmin()]
    print(f"\n\n\n\n threshold_levels: {threshold_levels}")
    target_val = pd.read_csv(model_location + tt + "_binary_target_val.csv").iloc[0]["Target-Val"]

    save_location = "../predictions-feb-update/"
    from pathlib import Path
    directory = Path(save_location)
    directory.mkdir(parents=True, exist_ok=True)

    print(f"stock: {ss}, target: {tt}, threshold: {threshold}, target_val: {target_val}")
    
    # Load the best performing model
    best_model = load_model(model_location + model_name)
    
    #data_frame = construct_values_for_model(ticker_symbols = stocks_to_check, sequence_size=20, use_for_last_day_prediction=True, verbose=False, refresh=True)
    data_frame, real_target = construct_values_for_model(ticker_symbols = [ss], 
                                                        underlying_target=tt,
                                                        sequence_size=dm_sequence_size(mm), 
                                                        use_for_last_day_prediction=False, 
                                                        verbose=False, 
                                                        data_interval="2y",
                                                        refresh=True,
                                                        is_binary_prediction=True,
                                                        incl_earnings=True
                                                        )

    print(f"all data : {data_frame}")
    data_frame = get_specific_date_data(data_frame, "2025-02-28")
    # data_frame = get_specific_date_internal_data(data_frame, "2025-01-01", "2025-01-30")

    i = True
    if i:
        print(data_frame)
        print(data_frame["LstmData"])
        if pd.isna(data_frame.iloc[-1]['Next-Day-Open']):
                print(f"ABORTING.... NEXT DAY'S DATA IS NOT AVAILABLE FOR THE LAST DAY IN YOUR SELECTION...")
                break
        i = False
    
    # print(data_frame)
    X_test = np.array(data_frame["LstmData"].to_list())
    X_test_ticker = np.array(data_frame["Ticker"].to_list())
    X_test_sector = np.array(data_frame["Sector"].to_list())
    X_test_yesterday = data_frame[lstm_features].to_numpy()
    X_test_days = np.array(data_frame["Date"].to_list())
    y_test = np.array(data_frame["y-value"].to_list())
    
    # print(X_test[0])

    # print(data_frame.iloc[0]["LstmData"].loc[:,"Date"])
    # for i in range(len(data_frame)):
    #         print(f"Stock: {X_test_ticker[i]}, Date: {data_frame['Date'].iloc[i]}, y-value: {data_frame['y-value'].iloc[i]}")

    y_predict = best_model.predict([X_test, X_test_ticker, X_test_sector, X_test_yesterday])
    data_frame['y-predict'] = y_predict
    for ticker in data_frame["Orig_Ticker"].unique():
        data_frame.loc[data_frame["Orig_Ticker"] == ticker, 'y-predict-original'] = inverse_normalize_data(data_frame[['y-predict']], real_target, ticker)

    data_frame["Target-Val"] = target_val
    data_frame["Threshold"] = threshold
    data_frame["Target"] = tt
    data_frame = data_frame.assign(**threshold_levels.to_dict())
    data_frame["Score"] = score
    data_frame["Mode"] = mm
    data_frame["Predicted-Price"] = data_frame["Next-Day-Open"] * (1 + data_frame["Target-Val"])
    data_frame["Close-To-Open-Pct"] = data_frame["Next-Day-Close"] / data_frame["Next-Day-Open"]
    data_frame["High-To-Open-Pct"] = data_frame["Next-Day-High-To-Next-Day-Open-Ratio"]
    data_frame["Low-To-Open-Pct"] = data_frame["Next-Day-Low-To-Next-Day-Open-Ratio"]
    data_frame["y-value-balanced"] = data_frame["Next-Day-High-To-Next-Day-Open-Ratio"] >= data_frame["Target-Val"]
    
    data_frame['y-predict-binary'] = data_frame['y-predict-original'] >= threshold
    data_frame['y-predict-binary-with-5-percent-grace'] = data_frame['y-predict-original'] >= threshold * 0.95
    print(f"PRINTING POSITIVE PREDICTIONS FOR: {ss} WITH TARGET: {tt}")
    print(data_frame[data_frame['y-predict-binary'] == 1.0])
    
    to_write = data_frame[["Orig_Ticker", "Date", "y-value", "y-value-balanced", "y-predict", "y-predict-original", "y-predict-binary", "y-predict-binary-with-5-percent-grace" ,"Score", "Mode", "Threshold", "TPs", "FPs", "FNs", "Precision", "Recall", "Random-Guess", "Target", "Target-Val", "Next-Day-Open", "Predicted-Price", "Next-Day-Close", "Close-To-Open-Pct", "High-To-Open-Pct", "Low-To-Open-Pct"]]
    to_write.to_csv(save_location + "todays-guess.csv", mode='a', index=False, header=write_header)
    
    to_write_only_true = to_write[to_write["y-predict-binary"] == 1.0]
    to_write_only_true.to_csv(save_location + "todays-guess_only_true.csv", mode='a', index=False, header=write_header)
    
    to_write_only_true = to_write[to_write["y-predict-binary-with-5-percent-grace"] == 1.0]
    to_write_only_true.to_csv(save_location + "todays-guess_only_true_5_percent_grace.csv", mode='a', index=False, header=write_header)
    write_header = False

SS: ABNB, MM: experimental TT: Next-Day-Close-To-Next-Day-Open-Ratio




 threshold_levels: threshold       0.477506
TPs             2.000000
FPs             1.000000
FNs             8.000000
Precision       0.666667
Recall          0.200000
Random-Guess    0.169492
Name: 3, dtype: float64
stock: ABNB, target: Next-Day-Close-To-Next-Day-Open-Ratio, threshold: 0.4775058627128601, target_val: 0.0144056947200816


2025-03-03 15:37:58.657342: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2025-03-03 15:37:58.657365: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-03-03 15:37:58.657371: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-03-03 15:37:58.657387: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-03-03 15:37:58.657397: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/keras/src/trainers/trainer.py:212: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=Fal

fetch_finance_data_for_tickers: about to load from yfinance: ABNB for interval: 2y
DOING EARNINGS
fixed_mean: -0.0004350308697668304, fixed_stddev: 0.015119581091662025


/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python

{'ABNB': 0}
{'Unknown': 0}
all data :                                               LstmData       Date  Ticker  \
0         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-04-30       0   
1         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-05-01       0   
2         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-05-02       0   
3         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-05-03       0   
4         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-05-06       0   
..                                                 ...        ...     ...   
205       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-02-25       0   
206       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-02-26       0   
207       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-02-27       0   
208       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-02-28       0   
209       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-03-03       0   

     Sector  Target-20D-Mean  Target-

...

fetch_finance_data_for_tickers: about to load from yfinance: DASH for interval: 2y
DOING EARNINGS
fixed_mean: 0.0007515638833232103, fixed_stddev: 0.01861324958807759


/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python

{'DASH': 0}
{'Consumer Tech': 0}
all data :                                               LstmData       Date  Ticker  \
0         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-02-06       0   
1         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-02-07       0   
2         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-02-08       0   
3         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-02-09       0   
4         High-Low-Ratio  High-Open-Ratio  Close-Op... 2024-02-12       0   
..                                                 ...        ...     ...   
263       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-02-25       0   
264       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-02-26       0   
265       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-02-27       0   
266       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-02-28       0   
267       High-Low-Ratio  High-Open-Ratio  Close-Op... 2025-03-03       0   

     Sector  Target-20D-Mean  T

/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator_v2.py:687: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_frame["Ticker"] = data_frame["Ticker"].replace(stock_dict)
/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator_v2.py:702: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_frame["Sector"] = data_frame["Sector"].replace(sector_dict)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
PRINTING POSITIVE PREDICTIONS FOR: DASH WITH TARGET: Next-Day-Close-To-Next-Day-Open-Ratio
Empty DataFrame
Columns: [LstmData, Date, Ticker, Sector, Target-20D-Mean, Target-20D-Sigma, y-value, High-Low-Ratio, High-Open-Ratio, Close-Open-Ratio, Close-Prev-Close, 200D-Ratio, 100D-Ratio, 20D-Ratio, MACD_HIST_NORM, MACD-Increase-Prev-MACD, MACD_BUY, MACD_SELL, Volatility-20, Volatility-100, Volatility-200, MACD_SIG, MFI3, RSI, RSI3, POCR, fastk, fastd, SlowK1, willr3, willr2, willr1, Next-Day-Open-To-Open, Next-Day-Open-To-Close, Today-Close-To-Open-Ratio-Above-One-Sigma, CMO3, CCI3, CCI2, ROC4, AroonOsc3, Is-Monday, Is-Tuesday, Is-Wednesday, Is-Thursday, Is-Friday, Is-Next-Trading-Day-Monday, Is-Next-Trading-Day-Tuesday, Is-Next-Trading-Day-Wednesday, Is-Next-Trading-Day-Thursday, Is-Next-Trading-Day-Friday, today_is_earning_day_evening, today_is_earning_day_morning, tomorrow_evening_earning_day, tomorrow_morning_earning_day, yesterday_evening_earnin

In [3]:
print(data_frame.iloc[0]["LstmData"])

     High-Low-Ratio  High-Open-Ratio  Close-Open-Ratio  Close-Prev-Close  \
467        0.347435         0.334434          0.328303          0.326475   
468        0.350642         0.338099          0.328060          0.337135   
469        0.346983         0.336956          0.326358          0.340656   
470        0.346798         0.338467          0.331078          0.332009   
471        0.341796         0.334633          0.333754          0.347498   
472        0.349041         0.342647          0.340943          0.347165   
473        0.343257         0.334558          0.325949          0.331761   
474        0.363448         0.353665          0.352354          0.353782   
475        0.342280         0.336248          0.332832          0.334025   
476        0.360097         0.351081          0.348465          0.330279   
477        0.359961         0.343928          0.340677          0.340677   
478        0.343387         0.340093          0.336470          0.337995   
479        0

In [4]:
data_frame = pd.read_csv(save_location + "todays-guess_only_true.csv")

filtered = data_frame[data_frame["Score"] >= 4]
filtered = filtered[filtered["y-predict-binary"] == True]
# Assign to the losses, that as if the market closed at the worst that the negative target value is hit.
filtered.loc[filtered['Close-To-Open-Pct'] < (1 - filtered['Target-Val']), 'Close-To-Open-Pct'] = (1 - filtered['Target-Val'])
filtered.loc[filtered['Close-To-Open-Pct'] < 0.985, 'Close-To-Open-Pct'] = 0.985
# filtered.loc[filtered['Low-To-Open-Pct'] < -0.0454, 'y-value-balanced'] = False
# # filtered.loc[filtered['Low-To-Open-Pct'] < -0.0454, 'Close-To-Open-Pct'] = 1-0.0454

gains = filtered[filtered["y-value-balanced"] == 1.0]["Target-Val"]
num_gains = gains.count()
total_gains = gains.sum()
num_loss = (filtered[filtered["y-value-balanced"] == 0.0]["Close-To-Open-Pct"] - 1.0).count()


losses = filtered[filtered["y-value-balanced"] == 0.0]

total_loss_uf = losses[losses["y-value-balanced"] == 0.0]["Close-To-Open-Pct"] - 1.0
total_loss = total_loss_uf.sum()

gains = gains+1
total_loss_uf = total_loss_uf + 1
# print(gains)

gains = (271.6504 * (gains**2)) - (519.3991 * (gains)) + 248.6636

# print(gains)
# print(f"\n\n\n\n\n")

# print(total_loss_uf)

losses = (271.6504 * (total_loss_uf**2)) -(519.3991 * (total_loss_uf)) + 248.6636

# print(losses)
# print(f"\n\n\n\n\n")

sum_ggg = gains.sum()
sum_lll = losses.sum()

#271.6504x2 −519.3991x+248.6636

print(f"{num_gains} Gains: {total_gains}, {num_loss} Loss: {total_loss}")
print(f"AVG Gain: {total_gains/ num_gains}, AVG Loss: {total_loss / num_loss}")
print(f"Total-GAIN: {sum_ggg}, Total-LOSS: {sum_lll}")
print(f"Started With: {num_loss + num_gains}, Ended with: {sum_ggg + sum_lll}, Calculated GAIN: {100 * ((sum_ggg + sum_lll) / (num_loss + num_gains)-1)}%")

0 Gains: 0.0, 2 Loss: 0.030999075248775654
AVG Gain: nan, AVG Loss: 0.015499537624387827
Total-GAIN: 0.0, Total-LOSS: 2.7012643379846395
Started With: 2, Ended with: 2.7012643379846395, Calculated GAIN: 35.063216899231975%


/var/folders/1n/8vt634f15059yz_p3_whxnk80000gn/T/ipykernel_22666/2434253093.py:44: RuntimeWarning: invalid value encountered in scalar divide
  print(f"AVG Gain: {total_gains/ num_gains}, AVG Loss: {total_loss / num_loss}")


In [5]:
money = 100
filtered = filtered.sort_values(by='Date', ascending=True)
filtered = filtered.reset_index(drop=True)

for day in filtered["Date"].unique():
    trades = filtered[filtered["Date"] == day]
    per_invest = money / max(4,len(trades))
    invest_result = money - (per_invest * len(trades))
    trades = trades.reset_index(drop=True) 

    # print(trades)
    trades = trades.loc[trades.groupby(['Orig_Ticker', 'Target'])['Score'].idxmax()]
    # print(trades)
    trades = trades.sample(frac=1).drop_duplicates(subset=['Orig_Ticker', 'Target'])
    # print(trades)

    
    trades = trades.loc[trades.groupby(['Orig_Ticker'])['Score'].idxmax()]
    trades = trades.loc[trades.groupby(['Orig_Ticker'])['Target-Val'].idxmax()]
    # print(trades)
    trades = trades.sample(frac=1).drop_duplicates(subset=['Orig_Ticker'])
    # print(trades)

    trades = trades.reset_index(drop=True) 

    for index, trade in trades.iterrows():
        if trade["y-value-balanced"] == True:
            target = trade["Target-Val"] + 1
            # print(f"Target: {target}")
            gain = (271.6504 * (target**2)) - (519.3991 * (target)) + 248.6636
            # print(f"gain: {gain}")
            invest_result += per_invest * gain
        else:
            target = trade["Close-To-Open-Pct"]
            print(f"Target: {target}")
            loss = (271.6504 * (target**2)) - (519.3991 * (target)) + 248.6636
            print(f"loss: {loss}")
            invest_result += per_invest * loss
    print(f"END OF DAY:{day} TRADES: {len(trades)} MONEY: {invest_result} STOCKS: {trades['Orig_Ticker'].unique()}")
    money = invest_result
        
print(money)

Target: 1.015341838967618
loss: 1.3455349366242046
END OF DAY:2025-02-28 TRADES: 1 MONEY: 83.63837341560512 STOCKS: ['BA']
83.63837341560512


In [6]:
x = [11,2,3,4,5,6,7,8,9,10]

df = pd.DataFrame({'x': x})

val = df['x'].rolling(window=5, min_periods=1).max().shift(-5)
print(val)

0     6.0
1     7.0
2     8.0
3     9.0
4    10.0
5     NaN
6     NaN
7     NaN
8     NaN
9     NaN
Name: x, dtype: float64


In [7]:
# data_frame["y-predict"] = y_predict

# sector_threshold_mapping =  pd.read_csv(model_location + model_name + ".csv")
# print(sector_threshold_mapping)
# for sector in data_frame["Orig_Sector"].unique():
#     threshold = sector_threshold_mapping[sector_threshold_mapping["Sector"] == sector].iloc[0]["Threshold"]
#     data_frame.loc[data_frame["Orig_Sector"] == sector, "y-predict-binary"] = (data_frame[data_frame["Orig_Sector"] == sector]["y-predict"] > threshold).astype(float)

# print(f"\n\nResults: \n {data_frame.loc[(data_frame['y-predict-binary'] == 1) & ((data_frame['Target-20D-Mean'] + data_frame['Target-20D-Sigma']) > 0.011)]}")


In [8]:
import numpy as np
X = np.load('X_values.npy')
y = np.load('y_values.npy')
dates = np.load('dates_values.npy', allow_pickle=True)
tickers = np.load('tickers_values.npy', allow_pickle=True)

FileNotFoundError: [Errno 2] No such file or directory: 'X_values.npy'

In [ ]:
import pandas as pd


index = 1100
print(sum(y))

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(X.shape)
    print(dates.shape)
    print(tickers.shape)
    arr = X[index,0:64,3]
    print(dates[index])
    print(tickers[index])
    print(y[index])
    for i in arr:
        print(i)
    # print(arr.min())